# Notebook 01: Multi-Tier Exploratory Data Analysis & Preprocessing

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/fahadrahman444/scientific-research-and-methodologies/blob/main/colab_notebooks/01_Bangladesh_Weather_Crop_EDA_and_Preprocessing.ipynb)

**Research Topic:** A Data-Driven Analysis of 15-Year Weather Variability and Its Impact on Agricultural Crop Yield in Bangladesh  
**Methodological Framework:** Two-Tier Quantitative Design (Option A)  
- **Tier 1 (Macro):** Official UN FAOSTAT National Rice Time Series (1976–2024, focusing on 2009–2024).  
- **Tier 2 (Sub-National):** 8 Administrative Divisions & 6 Staple/Commercial Crops (768 records, 2009–2024).  
**Course:** CSE 418: Scientific Research and Methodologies Lab (UITS)  
**Author:** Fahad Rahman | **Supervisor:** Lecturer Tahmina Islam  

---
## 1. Cloud Environment Setup


In [ ]:
# Step 1: Libraries and Aesthetics
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 11
print('Dependencies loaded successfully.')


In [ ]:
# Step 2: Automated Ingestion of Both Tiers (Colab Ready)
# Tier 1: National FAOSTAT Files
FAO_RAW_LOCAL = '../data/raw/FAOSTAT_data_en_10-12-2026.csv'
FAO_RAW_URL = f'{BASE_RAW}/data/raw/FAOSTAT_data_en_10-12-2026.csv'
INTEGRATED_LOCAL = '../data/processed/bangladesh_faostat_weather_integrated_15yr.csv'
INTEGRATED_URL = f'{BASE_RAW}/data/processed/bangladesh_faostat_weather_integrated_15yr.csv'

# Tier 2: Sub-National Divisional Multi-Crop File
REGIONAL_LOCAL = '../data/processed/bangladesh_weather_crop_15yr_processed.csv'
REGIONAL_URL = f'{BASE_RAW}/data/processed/bangladesh_weather_crop_15yr_processed.csv'

# Load Tier 1
df_fao_raw = pd.read_csv(FAO_RAW_LOCAL) if os.path.exists(FAO_RAW_LOCAL) else pd.read_csv(FAO_RAW_URL)
df_fao_integrated = pd.read_csv(INTEGRATED_LOCAL) if os.path.exists(INTEGRATED_LOCAL) else pd.read_csv(INTEGRATED_URL)

# Load Tier 2
df_regional = pd.read_csv(REGIONAL_LOCAL) if os.path.exists(REGIONAL_LOCAL) else pd.read_csv(REGIONAL_URL)

print(f'[Tier 1] Loaded FAOSTAT Raw: {len(df_fao_raw)} rows | 15-Yr Integrated: {len(df_fao_integrated)} years.')
print(f'[Tier 2] Loaded Regional Multi-Crop Dataset: {len(df_regional)} records across 8 divisions & 6 crops.')


## 2. Tier 1 (Macro Analysis): UN FAOSTAT 49-Year Evolution (1976–2024)
Examines long-term national trends in rice yield (kg/ha), production (million tonnes), and harvested area.


In [ ]:
# Step 3: 49-Year Historical Trajectory (FAOSTAT)
pvt_fao = df_fao_raw.pivot(index='Year', columns='Element', values='Value').reset_index()

fig, axes = plt.subplots(3, 1, figsize=(14, 11), sharex=True)
axes[0].plot(pvt_fao['Year'], pvt_fao['Yield'], color='#2ca02c', lw=2.5, marker='o', ms=3)
axes[0].axvspan(2009, 2024, color='gold', alpha=0.25, label='15-Year Study Window (2009-2024)')
axes[0].set_title('UN FAOSTAT: Bangladesh Rice Yield Trend (1976-2024)', fontsize=13, fontweight='bold')
axes[0].set_ylabel('Yield (kg/ha)')
axes[0].legend()

axes[1].plot(pvt_fao['Year'], pvt_fao['Production'] / 1e6, color='#1f77b4', lw=2.5, marker='s', ms=3)
axes[1].axvspan(2009, 2024, color='gold', alpha=0.25)
axes[1].set_title('Total National Rice Production (Million Tonnes)', fontsize=13, fontweight='bold')
axes[1].set_ylabel('Production (Million Tonnes)')

axes[2].plot(pvt_fao['Year'], pvt_fao['Area harvested'] / 1e6, color='#9467bd', lw=2.5, marker='^', ms=3)
axes[2].axvspan(2009, 2024, color='gold', alpha=0.25)
axes[2].set_title('Total Harvested Paddy Area (Million Hectares)', fontsize=13, fontweight='bold')
axes[2].set_xlabel('Year')
axes[2].set_ylabel('Area (Million ha)')

plt.tight_layout()
plt.show()


In [ ]:
# Step 4: 15-Year Climate Shocks vs National FAOSTAT Yield Anomalies
fig, axes = plt.subplots(2, 2, figsize=(15, 9))

# Heat Stress Days
axes[0, 0].bar(df_fao_integrated['year'], df_fao_integrated['heat_days'], color='#d62728', alpha=0.75)
axes[0, 0].set_title('Annual Heat Stress Days (Tmax >= 35°C)', fontweight='bold')
axes[0, 0].set_ylabel('Days Count')

# FAOSTAT Yield Rate
axes[0, 1].plot(df_fao_integrated['year'], df_fao_integrated['yield_mt_ha'], 'o-', color='#2ca02c', lw=2)
axes[0, 1].set_title('National Rice Yield (MT/ha)', fontweight='bold')
axes[0, 1].set_ylabel('MT / ha')

# SPI Drought Index
cols = ['#1f77b4' if v >= 0 else '#ff7f0e' for v in df_fao_integrated['spi']]
axes[1, 0].bar(df_fao_integrated['year'], df_fao_integrated['spi'], color=cols, alpha=0.85)
axes[1, 0].axhline(0, color='k', lw=1)
axes[1, 0].set_title('Standardized Precipitation Index (SPI-3)', fontweight='bold')
axes[1, 0].set_ylabel('Z-score')

# Climate Anomaly Residual
axes[1, 1].bar(df_fao_integrated['year'], df_fao_integrated['yield_climate_anomaly_pct'], color='#9467bd', alpha=0.8)
axes[1, 1].axhline(0, color='k', lw=1)
axes[1, 1].set_title('De-trended Climate-Induced Yield Anomaly (%)', fontweight='bold')
axes[1, 1].set_ylabel('Deviation (%)')

plt.tight_layout()
plt.show()


## 3. Tier 2 (Sub-National Analysis): 8 Administrative Divisions & 6 Crops
Evaluates localized agro-ecological conditions (Rajshahi Barind tract, Sylhet rainfall basins, coastal Khulna) and differences across crops (Aus, Aman, Boro Rice, Wheat, Potato, Jute).


In [ ]:
# Step 5: Divisional Weather Distributions Across Bangladesh
plt.figure(figsize=(14, 6))
sns.boxplot(data=df_regional, x='division', y='rainfall_total_mm', hue='crop_season', palette='Set2')
plt.title('Seasonal Rainfall Patterns by Division (Barind Drought vs Sylhet Basin)', fontsize=13, fontweight='bold')
plt.xlabel('Administrative Division')
plt.ylabel('Seasonal Rainfall (mm)')
plt.xticks(rotation=15)
plt.tight_layout()
plt.show()


In [ ]:
# Step 6: 15-Year Yield Trajectories for All 6 Commodities
fig, axes = plt.subplots(2, 3, figsize=(18, 10))
axes = axes.flatten()
crops = df_regional['crop'].unique()

for idx, c in enumerate(crops):
    sns.lineplot(data=df_regional[df_regional['crop'] == c], x='year', y='yield_mt_ha', hue='division', ax=axes[idx], legend=False)
    axes[idx].set_title(f'{c} Yield Trend (2009-2024)', fontweight='bold')
    axes[idx].set_xlabel('Year')
    axes[idx].set_ylabel('Yield (MT/ha)')

plt.tight_layout()
plt.show()


In [ ]:
# Step 7: Sub-National Correlation Heatmap
num_vars = ['temp_max_mean', 'temp_min_mean', 'rainfall_total_mm', 'humidity_mean', 'gdd', 'spi_3month', 'heat_stress_days', 'yield_mt_ha']
plt.figure(figsize=(9, 7))
sns.heatmap(df_regional[num_vars].corr(), annot=True, fmt='.2f', cmap='coolwarm', vmin=-1, vmax=1, square=True)
plt.title('Sub-National Pearson Correlation: Weather Features vs Crop Yield', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()
